# Early Academic Risk Prediction and Student Support System
## Notebook 01: Dataset Understanding & Initial Inspection

---

### Section 1: Project Introduction & Problem Framing

#### 1. What are we doing?
In this notebook, we are conducting the **Dataset Understanding** phase of our machine learning project using the **Open University Learning Analytics Dataset (OULAD)**. We inspect the structure, schemas, row counts, data types, missing values, relationships, and the target variable across all 7 CSV files in the dataset before performing any feature engineering or model training.

#### 2. Why are we doing it?
In professional Data Science and Machine Learning engineering, jumping straight into model training without thoroughly understanding the raw data is a recipe for failure. Raw data often contains subtle traps:
- **Target leakage**: Features that include information from the future (after the prediction moment).
- **Data anomalies & hidden formats**: Missing values disguised as question marks (`'?'`), unexpected negative numbers, or incorrect data types.
- **Relational complexity**: Data spread across multiple tables with composite primary keys and one-to-many relationships.
- **Grouping dependencies**: The same student enrolling in multiple courses, which can cause severe data leakage if training and testing splits are created randomly.

By systematically inspecting the data first, we build a solid foundation, prevent bugs, and make informed architectural decisions.

#### 3. What is our prediction objective?
Our goal is to build an **Early Academic Risk Prediction System** for undergraduate students. The system is designed as an early warning tool for instructors and academic advisors, allowing them to intervene with timely academic support (tutoring, advising, outreach) before a student fails.

We frame this as **Option A: Binary Classification**:
- **At Risk ($y = 1$)**: The student is at risk of failing the course presentation (`Fail`).
- **Not at Risk ($y = 0$)**: The student successfully completes the course presentation (`Pass` or `Distinction`).
- **Withdrawn Students (`Withdrawn`)**: Students who dropped out or unregistered. We will **not** automatically classify them as failures during initial analysis. Instead, we inspect when and why they withdrew (e.g., before the course started vs. midway) before making a principled modeling decision.

#### 4. Strict Machine Learning Guardrails for this Project
Throughout this project, we adhere to rigorous engineering standards:
1. **Traditional ML Only**: We will use interpretable, robust tabular algorithms (e.g., Logistic Regression, Random Forests, Gradient Boosted Decision Trees). No deep learning.
2. **Preventing Target Leakage**: We will define a strict prediction cutoff time ($T_{pred}$, such as Day 30 or Day 60). No information recorded after $T_{pred}$ will ever be used as an input feature.
3. **Student-Level Grouping**: We must prevent records from the same individual student appearing in both training and test sets.
4. **No Premature Cleaning**: We inspect missing values and anomalies here without deleting or modifying raw records.
5. **Class Imbalance Awareness**: We evaluate models using risk-sensitive metrics (Recall, Precision, PR-AUC, F-beta) rather than misleading raw accuracy.


---
### Section 2: Import Libraries

#### 1. What are we doing?
We import the foundational Python libraries needed for data inspection, tabular manipulation, and visualization.

#### 2. Why are we doing it?
Each library serves a specific, essential role in our exploratory workflow:
- **`pandas`**: The standard library for manipulating structured tabular data (DataFrames and Series).
- **`numpy`**: Provides fast mathematical operations, array structures, and numeric representations (such as `np.nan`).
- **`matplotlib.pyplot`**: The core plotting engine in Python for creating charts and figures.
- **`seaborn`**: A statistical visualization library built on top of Matplotlib that provides attractive default aesthetics and intuitive chart APIs.
- **`pathlib.Path`**: Python's modern, object-oriented file path library that handles file paths across Windows, macOS, and Linux seamlessly.

#### 3. What does the code do?
The code imports these libraries, sets up Seaborn's clean visualization theme (`whitegrid`), and configures standard plot dimensions.

#### 4. What output should I expect?
No visual output will be displayed. The cell should execute cleanly and show a checkmark or execution counter (e.g., `[1]`).

#### 5. How should I interpret the output?
A silent, successful execution confirms that all required packages are correctly installed inside our `.venv` virtual environment.

#### 6. What errors might occur and how can I fix them?
- **`ModuleNotFoundError: No module named 'pandas'` (or `seaborn`, etc.)**:
  - *Cause*: VS Code is running with the global Python interpreter instead of our project's `.venv`.
  - *Fix*: Click on the kernel selector at the top-right corner of the VS Code notebook window, choose **Select Kernel > Python Environments...**, and select the interpreter located at `.venv\Scripts\python.exe`.


In [ ]:
# Import standard data science and visualization libraries
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display options and plot styling
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("All required libraries imported successfully.")


---
### Section 3: Define Dataset Paths & Verify File Availability

#### 1. What are we doing?
We construct the file path to our raw dataset directory (`data/raw/oulad/`) using `pathlib.Path` and verify that each of the seven expected OULAD CSV files is physically present on disk.

#### 2. Why are we doing it?
Hardcoding absolute paths (e.g., `C:\Users\...`) breaks code when shared with others or deployed to different machines. Using `pathlib.Path` with relative directory traversal ensures our code runs consistently across Windows, Linux, and cloud environments. Verifying that all files exist *before* loading prevents confusing downstream runtime errors.

#### 3. What does the code do?
- Determines the project root dynamically, whether the notebook is executed from the project root or from inside the `notebooks/` subfolder.
- Points to the `data/raw/oulad/` folder.
- Iterates over a list of the 7 expected CSV files.
- Checks whether each file exists and prints its file size in megabytes.
- Raises a clear, descriptive warning if any file is missing.

#### 4. What output should I expect?
A list showing all 7 files confirmed with green checkmarks, along with their on-disk file sizes.

#### 5. How should I interpret the output?
- Notice the file sizes: most files are small (under 10 MB), but `studentVle.csv` is notably large (~432 MB).
- This alerts us in advance that `studentVle.csv` represents fine-grained clickstream activity and will require thoughtful memory handling.

#### 6. What errors might occur and how can I fix them?
- **`FileNotFoundError` or "File missing" alert**:
  - *Cause*: The OULAD CSV files are not in `data/raw/oulad/` or the directory name is misspelled.
  - *Fix*: Check your folder structure in the VS Code file explorer. Ensure `data/raw/oulad/` exists and contains the original 7 CSV files without modified filenames.


In [ ]:
# Dynamically resolve project root and raw dataset directory
# This ensures code runs seamlessly whether executed from project root or inside notebooks/
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
data_dir = project_root / "data" / "raw" / "oulad"

print(f"Project root resolved to: {project_root.resolve()}")
print(f"Raw data directory:       {data_dir.resolve()}")
print("-" * 65)

# Expected OULAD CSV files
expected_files = [
    "assessments.csv",
    "courses.csv",
    "studentAssessment.csv",
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentVle.csv",
    "vle.csv"
]

# Verify presence and file size of each dataset file
missing_files = []
for file_name in expected_files:
    file_path = data_dir / file_name
    if file_path.is_file():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"[FOUND] {file_name:<25} ({size_mb:.2f} MB)")
    else:
        print(f"[MISSING] {file_name:<25} <-- FILE NOT FOUND!")
        missing_files.append(file_name)

print("-" * 65)
if not missing_files:
    print("Verification complete: All 7 OULAD CSV files are present and ready to load.")
else:
    raise FileNotFoundError(f"Missing files detected: {missing_files}. Please check data/raw/oulad/.")


---
### Section 4: Load the CSV Files

#### 1. What are we doing?
We load each of the seven CSV files into its own dedicated pandas DataFrame using clean, standardized snake_case variable names:
- `assessments`
- `courses`
- `student_assessment`
- `student_info`
- `student_registration`
- `student_vle`
- `vle`

#### 2. Why are we doing it?
Loading raw data directly into DataFrames allows us to inspect, query, and analyze each table in memory. We keep the raw tables separate so we can understand their unique roles before considering any merging or feature engineering.

#### 3. What does the code do?
Calls `pd.read_csv()` for each file. For `student_vle`, we acknowledge its large size (over 10.6 million rows) and measure its loading time so you can observe the performance impact of large tabular datasets.

#### 4. What output should I expect?
A printed confirmation for each loaded DataFrame along with the elapsed time for loading `student_vle`.

#### 5. How should I interpret the output?
Loading the first six tables takes less than a second total. `student_vle` may take 3 to 6 seconds depending on your machine's CPU and disk speed. This is normal and expected for a ~430 MB CSV file containing ~10.65 million rows.

#### 6. What errors might occur and how can I fix them?
- **`MemoryError` on loading `student_vle`**:
  - *Cause*: System RAM is constrained (e.g., less than 4 GB of available physical RAM).
  - *Fix*: Close unneeded background applications. If memory remains tight, `student_vle` can be loaded in chunks (`pd.read_csv(..., chunksize=...)`) or with optimized integer types (`uint32`, `int16`). On standard developer machines with 8 GB+ RAM, standard `pd.read_csv` succeeds readily.


In [ ]:
import time

print("Loading OULAD CSV files into pandas DataFrames...")
start_time = time.time()

assessments = pd.read_csv(data_dir / "assessments.csv")
courses = pd.read_csv(data_dir / "courses.csv")
student_assessment = pd.read_csv(data_dir / "studentAssessment.csv")
student_info = pd.read_csv(data_dir / "studentInfo.csv")
student_registration = pd.read_csv(data_dir / "studentRegistration.csv")
vle = pd.read_csv(data_dir / "vle.csv")

# student_vle is ~432 MB on disk and contains over 10 million rows
print("Loading studentVle.csv (this may take a few seconds)...")
vle_start = time.time()
student_vle = pd.read_csv(data_dir / "studentVle.csv")
vle_elapsed = time.time() - vle_start

total_elapsed = time.time() - start_time
print(f"studentVle loaded in {vle_elapsed:.2f} seconds.")
print(f"All 7 datasets loaded successfully in {total_elapsed:.2f} seconds.")


---
### Section 5: Dataset Overview

#### 1. What are we doing?
For every DataFrame, we inspect:
- Its **shape** (number of rows and columns).
- Its **first 5 rows** (`.head(5)`).
- Its **column names**.
- Its **data types** (`.dtypes`).
- Its **in-memory RAM footprint**.

#### 2. Why are we doing it?
Understanding shapes and types prevents costly misunderstandings. For example:
- Are numbers stored as numeric types (`int64`, `float64`) or as text strings (`object`)?
- How many total observations do we have?
- What are the exact column names (e.g., `id_student` vs `student_id`)?

#### 3. What does the code do?
We first create a summary table that aggregates the dimensions and memory consumption of all 7 DataFrames. Then, we display the first 5 rows and data types of each table in a clean, readable layout. We deliberately avoid printing entire tables to keep the notebook responsive.

#### 4. What output should I expect?
1. A summary table showing rows, columns, and memory usage in MB for each DataFrame.
2. Formatted previews of the top 5 rows and column schemas for each table.

#### 5. How should I interpret the output?
Notice key observations immediately:
- **`student_info`** contains 32,593 records and 12 columns. This represents our core student enrollment entity.
- **`student_assessment`** contains 173,912 records.
- **`student_vle`** contains 10,655,280 records and consumes ~1.5 GB of RAM.
- **Data type surprise**: Look closely at `student_assessment['score']`, `assessments['date']`, and `student_registration['date_registration']`. Notice they have `dtype: object` (strings) instead of numeric types! Why would an assessment score or a date be an `object`? We investigate this critical anomaly in Section 6.

#### 6. What errors might occur and how can I fix them?
- **Output truncation**: If pandas truncates columns or rows, `pd.set_option('display.max_columns', None)` ensures all columns remain visible.


In [ ]:
# Dictionary mapping names to DataFrames for structured inspection
datasets = {
    "courses": courses,
    "assessments": assessments,
    "vle": vle,
    "student_info": student_info,
    "student_registration": student_registration,
    "student_assessment": student_assessment,
    "student_vle": student_vle
}

# 1. High-level summary of dimensions and memory usage
overview_rows = []
for name, df in datasets.items():
    mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    overview_rows.append({
        "Dataset": name,
        "Rows": f"{df.shape[0]:,}",
        "Columns": df.shape[1],
        "Memory (MB)": f"{mem_mb:.2f} MB"
    })

overview_df = pd.DataFrame(overview_rows)
print("=== OULAD High-Level Overview ===")
display(overview_df)


In [ ]:
# 2. Detailed inspection: First 5 rows and data types for each table
for name, df in datasets.items():
    print(f"\n{'='*70}")
    print(f"TABLE: {name.upper()} | Shape: {df.shape}")
    print(f"{'='*70}")
    print("Column Data Types:")
    for col, dtype in df.dtypes.items():
        print(f"  - {col:<28}: {dtype}")
    print("\nFirst 5 Rows:")
    display(df.head(5))


---
### Section 6: Data Quality Inspection (Missing Values, Placeholders & Duplicates)

#### 1. What are we doing?
We systematically examine each DataFrame for:
1. **Standard missing values** (`NaN` / `None` detected by `.isna()`).
2. **Hidden missing values**: Specifically, the string character `'?'`, which OULAD frequently uses as a missing value placeholder.
3. **Duplicate rows** across all columns (`.duplicated().sum()`).
4. **Descriptive statistics** (`.describe()`) to inspect min, max, and value ranges.
5. **Suspicious or unexpected values**.

> **Crucial Rule**: We do **not** remove, drop, or impute missing values or duplicates in this notebook. This is an exploratory inspection stage. Preprocessing decisions must be documented and executed carefully in subsequent stages to prevent data leakage.

#### 2. Why are we doing it?
In real-world data science, missing data is rarely polite enough to always show up as `np.nan`. If a CSV contains `'?'`, pandas reads the entire column as text (`object`). If you blindly run numeric operations or pass it to an ML model, your code will crash or, worse, silently misinterpret the data.

#### 3. What does the code do?
- Scans every column for both standard `NaN` values and string `'?'` placeholders.
- Combines them to calculate the true total missing count and percentage for every column.
- Checks for fully duplicated rows.
- Computes summary statistics.

#### 4. What output should I expect?
- A missing value breakdown identifying exactly which columns contain missing data and `'?'` characters.
- A duplicate count showing that 6 tables have 0 duplicate rows, while `student_vle` has 787,170 duplicate rows.
- Descriptive statistics revealing negative dates and score ranges.

#### 5. How should I interpret the output?
1. **Hidden missing values confirmed**:
   - `student_assessment['score']`: 173 rows have `'?'`. This is why `score` was loaded as an `object`!
   - `assessments['date']`: 11 rows have `'?'`. These represent final exams where no submission date is scheduled in the module definition.
   - `student_info['imd_band']`: 1,111 rows have `'?'` (Index of Multiple Deprivation, a measure of socioeconomic background).
   - `student_registration['date_registration']`: 45 rows have `'?'`.
   - `student_registration['date_unregistration']`: 22,521 rows have `'?'`. **Interpretation**: A student who does *not* unregister completes the course! The missingness here is not an error; it is highly informative: missing means the student remained registered.
   - `vle['week_from']` and `vle['week_to']`: 5,243 rows (~82%) have `'?'`.
2. **Duplicate rows in `student_vle`**:
   - `student_vle` has 787,170 full-row duplicates. Why? In clickstream logging, a student interacting with the same online resource multiple times on the same date can generate separate log events. In feature engineering, we will aggregate clicks (`sum_click`) per student and resource rather than keeping raw rows.
3. **Negative numbers in dates**:
   - Notice negative values in `date_registration` (e.g., `-159`) and `date` in `student_vle` (e.g., `-10`).
   - In OULAD, **Day 0 is the official course start date**. Negative dates represent activity or registrations that took place *before* the course officially started. This is valid domain behavior, not an error!

#### 6. What errors might occur and how can I fix them?
- **`TypeError` when calling `.describe()` on numeric columns**:
  - *Cause*: Columns containing `'?'` are objects, so numeric summaries (mean, std) will not automatically calculate.
  - *Fix*: In this inspection step, we inspect object columns using `.value_counts()` or `.describe(include='object')`. In future data preparation, we will convert `'?'` to `np.nan` and cast the column to numeric.


In [ ]:
# 1. Comprehensive Missing Value & '?' Inspection
missing_report = []

for name, df in datasets.items():
    for col in df.columns:
        std_nulls = int(df[col].isna().sum())
        # Check for '?' character in object columns
        if df[col].dtype == object:
            q_nulls = int((df[col] == '?').sum())
        else:
            q_nulls = 0
            
        total_missing = std_nulls + q_nulls
        total_rows = len(df)
        pct_missing = (total_missing / total_rows) * 100
        
        if total_missing > 0:
            missing_report.append({
                "Dataset": name,
                "Column": col,
                "Standard NaN": std_nulls,
                "'?' Placeholders": q_nulls,
                "Total Missing": total_missing,
                "% Missing": f"{pct_missing:.2f}%"
            })

missing_df = pd.DataFrame(missing_report)
print("=== Columns with Missing Data (Standard NaN or '?' Placeholders) ===")
display(missing_df)


In [ ]:
# 2. Duplicate Row Check across all 7 DataFrames
duplicate_report = []

for name, df in datasets.items():
    n_dupes = int(df.duplicated().sum())
    pct_dupes = (n_dupes / len(df)) * 100
    duplicate_report.append({
        "Dataset": name,
        "Total Rows": f"{len(df):,}",
        "Duplicate Rows": f"{n_dupes:,}",
        "% Duplicates": f"{pct_dupes:.2f}%"
    })

duplicate_df = pd.DataFrame(duplicate_report)
print("=== Duplicate Rows Inspection ===")
display(duplicate_df)


In [ ]:
# 3. Descriptive Statistics for Demographic & Numeric Tables
print("=== Descriptive Statistics: student_info (Numeric Columns) ===")
display(student_info.describe())

print("\n=== Descriptive Statistics: student_info (Categorical Columns) ===")
display(student_info.describe(include='object'))

print("\n=== Descriptive Statistics: courses ===")
display(courses.describe())


---
### Section 7: Understand Dataset Relationships & Key Architecture

#### 1. What are we doing?
We examine how the seven tables connect to one another. Specifically, we investigate the key identifiers:
- `id_student`: Unique student identifier.
- `code_module`: Course module code (e.g., 'AAA', 'BBB').
- `code_presentation`: Presentation semester/year (e.g., '2013J', '2014B' - 'B' stands for February presentation, 'J' stands for October).
- `id_assessment`: Assessment identifier.
- `id_site`: Virtual Learning Environment (VLE) resource identifier.

#### 2. Why are we doing it?
Understanding primary keys (unique identifiers for a row) and foreign keys (references to rows in other tables) is necessary before joining data.
- Making unverified assumptions about keys causes catastrophic bugs: e.g., accidental Cartesian products (blowing up row counts), duplicated student records, or distorted statistics.
- Crucially, investigating `id_student` will reveal whether a student appears only once or multiple times in the dataset.

#### 3. What does the code do?
- Tests uniqueness of keys in each table (`.duplicated().sum()`).
- Checks how many unique students exist vs. total enrollment rows.
- Verifies referential integrity between tables (ensuring child foreign keys match parent primary keys).

#### 4. What output should I expect?
Verification results displaying whether candidate keys are strictly unique, along with exact matching counts between tables.

#### 5. How should I interpret the output?
1. **Primary Key of `student_info`**:
   - `id_student` by itself is **not** unique (there are 28,785 unique students across 32,593 rows).
   - Some students enroll in multiple courses or repeat a presentation!
   - The composite key `(code_module, code_presentation, id_student)` is **100% unique** (0 duplicates). This represents a single **student-course enrollment**.
2. **Student-Level Grouping Requirement (Preventing Data Leakage)**:
   - Because 28,785 unique students account for 32,593 enrollments, some students appear in more than one row.
   - If we use a naive random train-test split, records of the *same student* could end up in both the training set and the test set!
   - The model could memorize student-specific attributes rather than learning generalizable risk patterns.
   - **Conclusion**: We must use student-grouped cross-validation (`GroupKFold` or `GroupShuffleSplit` on `id_student`).
3. **Primary Key of `courses`**:
   - The composite key `(code_module, code_presentation)` uniquely identifies each of the 22 module presentations.
4. **Primary Key of `assessments`**:
   - `id_assessment` is 100% unique (206 assessments).
5. **Primary Key of `vle`**:
   - `id_site` is 100% unique (6,364 resources).
6. **Primary Key of `student_assessment`**:
   - The composite key `(id_assessment, id_student)` is 100% unique (173,912 records).
7. **Referential Integrity**:
   - Every `(code_module, code_presentation, id_student)` in `student_registration` perfectly matches `student_info`.

#### 6. What errors might occur and how can I fix them?
- **Key mismatch**: If merging on `id_student` alone instead of `['code_module', 'code_presentation', 'id_student']`, the row count will explode due to many-to-many matches. Always merge student-level tables on the full composite key.


#### Entity Relationship Architecture (OULAD)

The diagram below illustrates how the 7 tables connect:

```text
               +-------------------------+
               |         courses         |
               +-------------------------+
               | PK: code_module         |
               | PK: code_presentation   |
               +------------+------------+
                            |
           +----------------+----------------+
           | 1:N                             | 1:N
           v                                 v
+-------------------------+       +-------------------------+
|       assessments       |       |           vle           |
+-------------------------+       +-------------------------+
| PK: id_assessment       |       | PK: id_site             |
| FK: code_module,        |       | FK: code_module,        |
|     code_presentation   |       |     code_presentation   |
+------------+------------+       +------------+------------+
             |                                 |
             | 1:N                             | 1:N
             v                                 v
+-------------------------+       +-------------------------+
|   studentAssessment     |       |       studentVle        |
+-------------------------+       +-------------------------+
| FK: id_assessment       |       | FK: id_site             |
| FK: id_student          |       | FK: code_module,        |
+------------+------------+       |     code_presentation,  |
             |                    |     id_student          |
             | N:1                +------------+------------+
             |                                 | N:1
             +----------------+----------------+
                              |
                              v
               +-------------------------+
               |       studentInfo       | <=== TARGET TABLE
               +-------------------------+
               | PK: code_module         |
               | PK: code_presentation   |
               | PK: id_student          |
               | Target: final_result    |
               +------------+------------+
                            | 1:1
                            v
               +-------------------------+
               |   studentRegistration   |
               +-------------------------+
               | PK: code_module         |
               | PK: code_presentation   |
               | PK: id_student          |
               | date_registration       |
               | date_unregistration     |
               +-------------------------+
```


In [ ]:
# Programmatic Verification of Candidate Keys and Relationships

print("=== 1. student_info Keys ===")
total_student_rows = len(student_info)
unique_students = student_info['id_student'].nunique()
print(f"Total student_info rows:             {total_student_rows:,}")
print(f"Unique student IDs (id_student):     {unique_students:,}")
print(f"Students with multiple enrollments:  {total_student_rows - unique_students:,}")

st_composite = student_info[['code_module', 'code_presentation', 'id_student']]
dupes_st_composite = st_composite.duplicated().sum()
print(f"Duplicates on (code_module, code_presentation, id_student): {dupes_st_composite} -> "
      f"{'PERFECT PRIMARY KEY' if dupes_st_composite == 0 else 'NOT UNIQUE'}")

print("\n=== 2. student_registration Alignment ===")
reg_composite = student_registration[['code_module', 'code_presentation', 'id_student']]
dupes_reg_composite = reg_composite.duplicated().sum()
print(f"Duplicates on (code_module, code_presentation, id_student) in registration: {dupes_reg_composite} -> "
      f"{'PERFECT 1:1 ALIGNMENT' if dupes_reg_composite == 0 and len(student_registration) == total_student_rows else 'MISMATCH'}")

print("\n=== 3. courses Keys ===")
course_key = courses[['code_module', 'code_presentation']]
dupes_courses = course_key.duplicated().sum()
print(f"Total courses: {len(courses)}, Duplicates on (code_module, code_presentation): {dupes_courses} -> "
      f"{'PERFECT PRIMARY KEY' if dupes_courses == 0 else 'NOT UNIQUE'}")

print("\n=== 4. assessments Keys ===")
unique_assessments = assessments['id_assessment'].nunique()
print(f"Total assessments: {len(assessments)}, Unique id_assessment: {unique_assessments} -> "
      f"{'PERFECT PRIMARY KEY' if unique_assessments == len(assessments) else 'NOT UNIQUE'}")

print("\n=== 5. vle Keys ===")
unique_sites = vle['id_site'].nunique()
print(f"Total vle sites: {len(vle)}, Unique id_site: {unique_sites} -> "
      f"{'PERFECT PRIMARY KEY' if unique_sites == len(vle) else 'NOT UNIQUE'}")

print("\n=== 6. student_assessment Keys ===")
st_ass_key = student_assessment[['id_assessment', 'id_student']]
dupes_st_ass = st_ass_key.duplicated().sum()
print(f"Total student_assessment records: {len(student_assessment):,}")
print(f"Duplicates on (id_assessment, id_student): {dupes_st_ass} -> "
      f"{'PERFECT COMPOSITE KEY' if dupes_st_ass == 0 else 'NOT UNIQUE'}")


---
### Section 8: Understand the Target Variable (`final_result`)

#### 1. What are we doing?
We examine the distribution of the target variable `final_result` located in `student_info`.
We calculate:
- Category counts.
- Percentage distributions.
- A clean, informative bar chart.
We then conduct a deep pedagogical discussion on how these categories map to our initial **Binary Classification** problem: predicting whether a student is **at risk of failing**.

#### 2. Why are we doing it?
The target variable defines what our machine learning model learns to predict.
- If the target is imbalanced (e.g., 20% vs 80%), standard accuracy is meaningless.
- If we misunderstand what each category means—especially `Withdrawn`—we risk training a model on false assumptions.

#### 3. What does the code do?
- Computes frequencies and percentages of each category in `student_info['final_result']`.
- Generates a styled Seaborn bar plot annotated with both exact counts and percentage labels.

#### 4. What output should I expect?
A summary table and a bar plot showing the 4 distinct outcomes:
1. `Pass`
2. `Withdrawn`
3. `Fail`
4. `Distinction`

#### 5. How should I interpret the output?
Let's analyze the exact numbers:
- **`Pass`**: 12,361 (37.93%)
- **`Withdrawn`**: 10,156 (31.16%)
- **`Fail`**: 7,052 (21.64%)
- **`Distinction`**: 3,024 (9.28%)

Total student-course enrollments: **32,593**.

#### Critical Analysis: The Target Mapping & The "Withdrawn" Dilemma
Our project objective is **Option A: Binary Classification (Predict whether a student is at risk of failing)**.
- **Fail ($y = 1$)**: These students completed the course presentations and exams but did not achieve passing marks. They clearly represent **Academic Risk**.
- **Pass / Distinction ($y = 0$)**: These students completed the course successfully. They represent **Not at Risk**.
- **What about Withdrawn (31.16% of the dataset)?**
  - Notice that nearly a third of all enrollments result in withdrawal!
  - In educational analytics, withdrawal can stem from many non-academic causes: financial strain, illness, career changes, or dropping out before the first day of class.
  - Some students withdraw at day `-30` (before class even begins), while others withdraw at day `120` after failing early assignments.
  - If we automatically lump all `Withdrawn` into `Fail`, our model may attempt to predict life dropouts using early academic quiz scores, distorting the signal.
  - If we exclude `Withdrawn` during initial analysis, we evaluate only the **completing cohort** (22,437 students):
    - `Fail`: 7,052 (31.43%)
    - `Pass` + `Distinction`: 15,385 (68.57%)
  - **Decision**: In this notebook, we document this exact distribution and keep `Withdrawn` separate. We do **not** create a finalized binary target column yet. The final mapping and cutoff filtering will be established during exploratory data analysis once we inspect withdrawal dates in `student_registration`.

#### Class Imbalance Implications:
In the completing cohort, the minority class (`Fail`) represents ~31.4% of the population.
- A dummy baseline that always guesses "Not at Risk" would achieve ~68.6% accuracy while failing to identify a single student who needs help!
- Therefore, accuracy is an inadequate metric. For our support system, **Recall for Class 1 (At Risk)** is paramount: our priority is to catch as many struggling students as possible so the institution can intervene.

#### 6. What errors might occur and how can I fix them?
- **Warning on Seaborn palette**: In newer Seaborn versions, assigning `x` without `hue` or `legend=False` can trigger a deprecation note. The code below explicitly sets parameters to avoid warnings.


In [ ]:
# Target variable frequency and percentage distribution
target_counts = student_info['final_result'].value_counts()
target_pcts = (student_info['final_result'].value_counts(normalize=True) * 100).round(2)

target_summary = pd.DataFrame({
    "Category": target_counts.index,
    "Count": target_counts.values,
    "Percentage (%)": target_pcts.values
})

print("=== Target Variable Distribution (final_result) ===")
display(target_summary)

# Visualize the distribution with annotated bar plot
plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=target_summary,
    x="Category",
    y="Count",
    palette=["#4C72B0", "#C44E52", "#55A868", "#8172B2"]
)

plt.title("Distribution of Student Course Outcomes (final_result)", fontsize=14, weight='bold', pad=15)
plt.xlabel("Course Final Result", fontsize=12)
plt.ylabel("Number of Student Enrollments", fontsize=12)

# Annotate each bar with count and percentage
total_records = len(student_info)
for p in ax.patches:
    height = p.get_height()
    pct = (height / total_records) * 100
    ax.annotate(f"{int(height):,}\n({pct:.1f}%)",
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center',
                fontsize=11, color='white', weight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Inspecting Completing Cohort (Excluding Withdrawn) vs Full Cohort
completing_cohort = student_info[student_info['final_result'] != 'Withdrawn']
completing_counts = completing_cohort['final_result'].value_counts()
completing_pcts = (completing_cohort['final_result'].value_counts(normalize=True) * 100).round(2)

completing_summary = pd.DataFrame({
    "Category": completing_counts.index,
    "Count": completing_counts.values,
    "Percentage among Completers (%)": completing_pcts.values
})

print("=== Outcome Distribution for Students Completing the Course (n = 22,437) ===")
display(completing_summary)

at_risk_count = completing_counts.get('Fail', 0)
not_at_risk_count = completing_counts.get('Pass', 0) + completing_counts.get('Distinction', 0)
print(f"\nIf binary target is defined on completers only:")
print(f"  - At Risk (Fail = 1):         {at_risk_count:,} ({at_risk_count/len(completing_cohort)*100:.2f}%)")
print(f"  - Not at Risk (Pass/Dist = 0): {not_at_risk_count:,} ({not_at_risk_count/len(completing_cohort)*100:.2f}%)")
print(f"  - Ratio (Not at Risk : At Risk): {not_at_risk_count / at_risk_count:.2f} : 1")


---
### Section 9: Initial Findings & Synthesis

#### 1. What are we doing?
We synthesize the observations and discoveries made throughout this initial dataset inspection into a structured summary.

#### 2. Why are we doing it?
Clear documentation is what separates ad-hoc tinkering from professional data science. Summarizing findings here provides an anchor for all future feature engineering and modeling decisions.

#### 3. Summary of Key Findings

| Area | Observation & Evidence | Engineering / Modeling Implication |
| :--- | :--- | :--- |
| **Dataset Scale** | 7 CSV files; 32,593 enrollments across 22 module presentations; 28,785 unique students; 10.65M VLE interactions. | Small demographic tables, but clickstream interactions (`student_vle`) are large. Must aggregate clickstream data into summary metrics rather than joining raw logs. |
| **Primary Keys** | `student_info`: `(code_module, code_presentation, id_student)`. `courses`: `(code_module, code_presentation)`. `assessments`: `id_assessment`. `vle`: `id_site`. | Joining student data must always use the 3-part composite key to avoid accidental row duplication. |
| **Student Grouping** | 28,785 unique students generate 32,593 enrollments (3,808 students take multiple courses). | **Critical anti-leakage rule**: We must use `GroupKFold` or `GroupShuffleSplit` on `id_student` so records of the same student do not leak across train and test sets. |
| **Hidden Missing Values** | `'?'` strings found in `imd_band` (1,111), `score` (173), `date` (11), `date_registration` (45), `date_unregistration` (22,521), `week_from`/`week_to` (5,243). | All `'?'` strings must be replaced with `np.nan` and columns cast to proper numeric types during preprocessing. |
| **Duplicate Records** | 6 tables have 0 duplicates. `student_vle` has 787,170 full-row duplicates (7.39%). | In `student_vle`, students interact repeatedly with the same site on the same day. These must be aggregated (`sum_click`) by student and site before modeling. |
| **Target Distribution** | Full cohort: `Pass` (37.9%), `Withdrawn` (31.2%), `Fail` (21.6%), `Distinction` (9.3%). Completing cohort: `Not at Risk` (68.6%) vs `At Risk` (31.4%). | Moderate class imbalance. Models must be evaluated using Recall, Precision, PR-AUC, and F-beta rather than simple Accuracy. |
| **Withdrawn Status** | 31.16% of enrollments withdraw. Some unregister before day 0; others unregister mid-term. | Must be analyzed in depth during EDA alongside `date_unregistration` before finalizing target inclusion rules. |
| **Negative Dates** | Negative numbers in `date_registration` (e.g. -159) and `student_vle` (e.g. -10). | Not an error. Day 0 is course start date; negative values represent pre-course registration and pre-course VLE orientation activity. |
| **VLE Activity Scope** | VLE clicks track clicks on virtual learning pages (resources, forums, quizzes). | **Do not treat VLE clicks as physical classroom attendance.** It represents online engagement, which has different dynamics. |


---
### Section 10: Next Steps (Roadmap to Modeling)

#### 1. What are we doing?
We outline the concrete, structured phases that must be completed before any model training begins.

#### 2. Why are we doing it?
A structured roadmap ensures we do not skip necessary validation, leakage prevention, or feature engineering steps.

#### 3. Planned Progression:

```text
[01. Dataset Understanding] <--- (We are here!)
             │
             ▼
[02. Exploratory Data Analysis (EDA) & Cutoff Time Formulation]
  - Investigate demographic correlations with failure.
  - Analyze early submission scores and submission timing.
  - Define strict prediction cutoff time (T_pred, e.g., Day 30 or Day 60).
  - Formalize handling of Withdrawn students relative to T_pred.
             │
             ▼
[03. Feature Engineering & Anti-Leakage Aggregation]
  - Aggregate VLE clicks strictly BEFORE T_pred (sum, frequency, trend).
  - Aggregate assessment scores strictly BEFORE T_pred.
  - Create student demographic encodings.
  - Construct leak-free feature matrix X and target vector y.
             │
             ▼
[04. Validation Strategy & Baseline Modeling]
  - Implement GroupKFold on id_student.
  - Establish a dummy/heuristic baseline.
  - Train interpretable traditional models (Logistic Regression, Random Forest, LightGBM/XGBoost).
  - Evaluate using Recall, Precision, PR-AUC, and Confusion Matrix.
             │
             ▼
[05. Model Interpretation, Error Analysis & Support Decision Logic]
  - Feature importance and SHAP analysis.
  - Define probability thresholds for triggering teacher interventions.
  - Build Streamlit application for end-user teachers.
```

---
**End of Notebook 01: Dataset Understanding.**
*Please review this notebook, run the cells in your VS Code environment, and inspect the outputs before moving to EDA.*
